# Member 2: QLoRA + DoRA + Hyperparameter Experiments

**Author:** Wanrong Dang (wd2423)  
**Weeks:** 5-8  
**Goal:** Build on Member 1's LoRA pipeline and complete QLoRA, DoRA, and hyperparameter ablations

## Experiment Matrix

| Experiment | rank | 4-bit | DoRA | Estimated Time (T4) |
|---|---|---|---|---|
| QLoRA r=4 | 4 | ✅ | ❌ | ~3h |
| QLoRA r=8 (main experiment) | 8 | ✅ | ❌ | ~3h |
| QLoRA r=16 | 16 | ✅ | ❌ | ~3h |
| DoRA r=4 | 4 | ❌ | ✅ | ~3.5h |
| DoRA r=8 (main experiment) | 8 | ❌ | ✅ | ~3.5h |
| DoRA r=16 | 16 | ❌ | ✅ | ~3.5h |
| Q-DoRA r=8 (optional) | 8 | ✅ | ✅ | ~3h |

**Important:** Each experiment needs an independent Colab session. A T4 runtime should run only one experiment at a time. Use the `EXPERIMENT` variable to switch runs.

**Compute-budget recommendation:** main experiment (QLoRA r=8 + DoRA r=8) first, then r=4, and finally r=16.

## Step 1: Environment setup (~2-3 min)

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Clone Member 1's repository
!git clone https://github.com/qiujunzhang03-7/GR5293-peft-medical-vqa.git
%cd GR5293-peft-medical-vqa

In [ ]:
# Install dependencies (Colab already preinstalls torch, so do not reinstall it)
%pip install -q -r requirements.txt

# bitsandbytes is required for QLoRA; verify it separately
%pip install -q -U bitsandbytes

# qwen-vl-utils handles image tokens
%pip install -q -U qwen-vl-utils

In [ ]:
# Make the project importable in Python
import os, sys
PROJECT_ROOT = '/content/GR5293-peft-medical-vqa'
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f"CWD: {os.getcwd()}")

In [ ]:
# Smoke test: verify that the environment works (HANDOFF.md expects 85 passed)
!python -m pytest tests/ -q

In [ ]:
# Verify that bitsandbytes 4-bit quantization is available (required for QLoRA)
import torch
import bitsandbytes as bnb
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"bitsandbytes: {bnb.__version__}")

# Verify PEFT DoRA support
import peft
from peft import LoraConfig
print(f"peft: {peft.__version__}")
# Test whether the DoRA argument is recognized
test_cfg = LoraConfig(r=8, lora_alpha=16, target_modules=['q_proj'], use_dora=True)
print(f"DoRA support: ✅ (use_dora={test_cfg.use_dora})")

## Step 2: Upload configuration files (use Member 2 YAML files)

Put the 6 YAML files provided into the `configs/` directory:
- `qlora_rank4.yaml`, `qlora_rank8.yaml`, `qlora_rank16.yaml`
- `dora_rank4.yaml`, `dora_rank8.yaml`, `dora_rank16.yaml`

**Option A:** Create them directly below with `%%writefile` (recommended to avoid losing local files)  
**Option B:** Drag and drop them using the left-side Colab file browser

In [ ]:
%%writefile configs/qlora_rank8.yaml
model_id: "Qwen/Qwen2-VL-2B-Instruct"
dtype: "float16"
load_in_4bit: true
train_split: "train"
train_max_examples: null
eval_split: "test"
eval_max_examples: null
lora_r: 8
lora_alpha: 16
lora_dropout: 0.05
target_modules:
  - q_proj
  - k_proj
  - v_proj
  - o_proj
  - gate_proj
  - up_proj
  - down_proj
use_dora: false
num_epochs: 3
per_device_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 1.0e-4
weight_decay: 0.0
warmup_ratio: 0.03
max_grad_norm: 1.0
gradient_checkpointing: true
logging_steps: 25
output_dir: "checkpoints/qlora_rank8"
seed: 42
run_eval_after_training: true
eval_max_new_tokens: 64

In [ ]:
%%writefile configs/qlora_rank4.yaml
model_id: "Qwen/Qwen2-VL-2B-Instruct"
dtype: "float16"
load_in_4bit: true
train_split: "train"
train_max_examples: null
eval_split: "test"
eval_max_examples: null
lora_r: 4
lora_alpha: 8
lora_dropout: 0.05
target_modules:
  - q_proj
  - k_proj
  - v_proj
  - o_proj
  - gate_proj
  - up_proj
  - down_proj
use_dora: false
num_epochs: 3
per_device_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 1.0e-4
weight_decay: 0.0
warmup_ratio: 0.03
max_grad_norm: 1.0
gradient_checkpointing: true
logging_steps: 25
output_dir: "checkpoints/qlora_rank4"
seed: 42
run_eval_after_training: true
eval_max_new_tokens: 64

In [ ]:
%%writefile configs/qlora_rank16.yaml
model_id: "Qwen/Qwen2-VL-2B-Instruct"
dtype: "float16"
load_in_4bit: true
train_split: "train"
train_max_examples: null
eval_split: "test"
eval_max_examples: null
lora_r: 16
lora_alpha: 32
lora_dropout: 0.05
target_modules:
  - q_proj
  - k_proj
  - v_proj
  - o_proj
  - gate_proj
  - up_proj
  - down_proj
use_dora: false
num_epochs: 3
per_device_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 1.0e-4
weight_decay: 0.0
warmup_ratio: 0.03
max_grad_norm: 1.0
gradient_checkpointing: true
logging_steps: 25
output_dir: "checkpoints/qlora_rank16"
seed: 42
run_eval_after_training: true
eval_max_new_tokens: 64

In [ ]:
%%writefile configs/dora_rank8.yaml
model_id: "Qwen/Qwen2-VL-2B-Instruct"
dtype: "float16"
load_in_4bit: false
train_split: "train"
train_max_examples: null
eval_split: "test"
eval_max_examples: null
lora_r: 8
lora_alpha: 16
lora_dropout: 0.05
target_modules:
  - q_proj
  - k_proj
  - v_proj
  - o_proj
  - gate_proj
  - up_proj
  - down_proj
use_dora: true
num_epochs: 3
per_device_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 1.0e-4
weight_decay: 0.0
warmup_ratio: 0.03
max_grad_norm: 1.0
gradient_checkpointing: true
logging_steps: 25
output_dir: "checkpoints/dora_rank8"
seed: 42
run_eval_after_training: true
eval_max_new_tokens: 64

In [ ]:
%%writefile configs/dora_rank4.yaml
model_id: "Qwen/Qwen2-VL-2B-Instruct"
dtype: "float16"
load_in_4bit: false
train_split: "train"
train_max_examples: null
eval_split: "test"
eval_max_examples: null
lora_r: 4
lora_alpha: 8
lora_dropout: 0.05
target_modules:
  - q_proj
  - k_proj
  - v_proj
  - o_proj
  - gate_proj
  - up_proj
  - down_proj
use_dora: true
num_epochs: 3
per_device_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 1.0e-4
weight_decay: 0.0
warmup_ratio: 0.03
max_grad_norm: 1.0
gradient_checkpointing: true
logging_steps: 25
output_dir: "checkpoints/dora_rank4"
seed: 42
run_eval_after_training: true
eval_max_new_tokens: 64

In [ ]:
%%writefile configs/dora_rank16.yaml
model_id: "Qwen/Qwen2-VL-2B-Instruct"
dtype: "float16"
load_in_4bit: false
train_split: "train"
train_max_examples: null
eval_split: "test"
eval_max_examples: null
lora_r: 16
lora_alpha: 32
lora_dropout: 0.05
target_modules:
  - q_proj
  - k_proj
  - v_proj
  - o_proj
  - gate_proj
  - up_proj
  - down_proj
use_dora: true
num_epochs: 3
per_device_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 1.0e-4
weight_decay: 0.0
warmup_ratio: 0.03
max_grad_norm: 1.0
gradient_checkpointing: true
logging_steps: 25
output_dir: "checkpoints/dora_rank16"
seed: 42
run_eval_after_training: true
eval_max_new_tokens: 64

## Step 3: Select and run an experiment

**Important:** Change the `EXPERIMENT` variable below to choose which experiment to run, then run Step 4.

Recommended order:
1. First run `qlora_smoke` (Smoke test, 5 min, verify that 4-bit loading works)
2. Then run `qlora_rank8` (main experiment, ~3h)
3. Then run `dora_rank8` (main experiment, ~3.5h)
4. Finally run r=4 and r=16 to complete the ablation

In [ ]:
# >>>>> Change the experiment to run here <<<<<
EXPERIMENT = 'qlora_smoke'

# Allowed values:
# 'qlora_smoke'   - Smoke test (50 samples, 1 epoch) - 5 minutes
# 'dora_smoke'    - DoRA smoke test
# 'qlora_rank4'
# 'qlora_rank8'
# 'qlora_rank16'
# 'dora_rank4'
# 'dora_rank8'
# 'dora_rank16'
# 'qdora_rank8'   - optional: Q-DoRA combined run

print(f"Preparing to run experiment: {EXPERIMENT}")

## Step 4: Run training

**Note:** Full training (~3 hours) keep the browser active; otherwise Colab may disconnect.  
**After disconnection:** If the session disconnects, saved epoch checkpoints are not lost, but this pipeline must be rerun from the beginning (this pipeline does not support resume).

In [ ]:
# Smoke test: Run 50 samples for 1 epoch to verify the end-to-end pipeline
if EXPERIMENT == 'qlora_smoke':
    !python -m src.training.train_lora \
        --config configs/qlora_rank8.yaml \
        --max_train 50 \
        --epochs 1 \
        --output_dir checkpoints/qlora_smoke
elif EXPERIMENT == 'dora_smoke':
    !python -m src.training.train_lora \
        --config configs/dora_rank8.yaml \
        --max_train 50 \
        --epochs 1 \
        --output_dir checkpoints/dora_smoke
else:
    print(f"Not a smoke test; skip this cell. Run the next cell for the full experiment.")

In [ ]:
# Full training: Select the matching YAML based on EXPERIMENT
import shlex, subprocess, time

config_map = {
    'qlora_rank4':  'configs/qlora_rank4.yaml',
    'qlora_rank8':  'configs/qlora_rank8.yaml',
    'qlora_rank16': 'configs/qlora_rank16.yaml',
    'dora_rank4':   'configs/dora_rank4.yaml',
    'dora_rank8':   'configs/dora_rank8.yaml',
    'dora_rank16':  'configs/dora_rank16.yaml',
    'qdora_rank8':  'configs/qdora_rank8.yaml',
}

if EXPERIMENT in config_map:
    cfg = config_map[EXPERIMENT]
    print(f"=== Starting training: {EXPERIMENT} → {cfg} ===")
    print(f"Start time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Estimated duration: ~3-3.5 hours\n")
    
    !python -m src.training.train_lora --config {cfg}
    
    print(f"\n=== Training finished at: {time.strftime('%Y-%m-%d %H:%M:%S')} ===")
elif EXPERIMENT in ('qlora_smoke', 'dora_smoke'):
    print("The smoke test was completed in the previous cell.")
else:
    raise ValueError(f"Unknown EXPERIMENT: {EXPERIMENT}")

## Step 5: Verify training results

Verify each item using the HANDOFF.md § 16 checklist.

In [ ]:
import json
from pathlib import Path

# Automatically locate the checkpoint from the run
if EXPERIMENT.endswith('_smoke'):
    ckpt_dir = Path(f"checkpoints/{EXPERIMENT}")
else:
    ckpt_dir = Path(f"checkpoints/{EXPERIMENT}")

print(f"Checkpoint directory: {ckpt_dir}")
print(f"Files included:")
for f in sorted(ckpt_dir.iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:<35} {size_mb:>8.2f} MB")

In [ ]:
# HANDOFF.md § 16 final-results checklist
print("=" * 60)
print(f"Final Results Checklist: {EXPERIMENT}")
print("=" * 60)

checks = []

# 1. training_metrics.json
metrics_file = ckpt_dir / "training_metrics.json"
if metrics_file.exists():
    with open(metrics_file) as f:
        m = json.load(f)
    checks.append(("training_metrics.json exists", True))
    
    trainable = m['params']['trainable']
    pct = m['params']['trainable_pct']
    peak_gpu = m['peak_gpu_memory_gb']
    total_sec = m['training']['total_seconds']
    
    print(f"\n--- Training statistics ---")
    print(f"Method:                {m['method']}")
    print(f"trainable parameters:          {trainable:,} ({pct:.4f}%)")
    print(f"Peak GPU memory:        {peak_gpu:.2f} GB")
    print(f"Total training time:          {total_sec:.0f} s ({total_sec/3600:.2f} h)")
    
    if 'evaluation' in m:
        ev = m['evaluation']
        print(f"\n--- Test-set Evaluation (n=451) ---")
        print(f"Closed EM (n={ev['closed']['n']}):     {ev['closed']['exact_match']:.4f}")
        if 'ci95' in ev['closed']:
            ci = ev['closed']['ci95']
            print(f"  95% CI: [{ci['lower']:.4f}, {ci['upper']:.4f}]")
        
        print(f"Open Token-F1 (n={ev['open']['n']}):  {ev['open']['f1']:.4f}")
        if 'ci95_f1' in ev['open']:
            ci = ev['open']['ci95_f1']
            print(f"  95% CI: [{ci['lower']:.4f}, {ci['upper']:.4f}]")
        
        print(f"Overall EM (n={ev['overall']['n']}):  {ev['overall']['exact_match']:.4f}")
        if 'ci95' in ev['overall']:
            ci = ev['overall']['ci95']
            print(f"  95% CI: [{ci['lower']:.4f}, {ci['upper']:.4f}]")
else:
    checks.append(("training_metrics.json exists", False))

# 2. per_example_scores.json (needed for cross-method paired statistical tests)
scores_file = ckpt_dir / "per_example_scores.json"
if scores_file.exists():
    with open(scores_file) as f:
        s = json.load(f)
    n_ids = len(s['ids'])
    checks.append(("per_example_scores.json exists", True))
    checks.append((f"per_example_scores contains 451 samples (actual: {n_ids})", n_ids == 451))
else:
    checks.append(("per_example_scores.json exists", False))

# 3. lora_predictions.jsonl
pred_file = ckpt_dir / "lora_predictions.jsonl"
if pred_file.exists():
    n_lines = sum(1 for _ in open(pred_file))
    checks.append((f"lora_predictions.jsonl contains 451 lines (actual: {n_lines})", n_lines == 451))
else:
    checks.append(("lora_predictions.jsonl exists", False))

# 4. adapter_model.safetensors
adapter_file = ckpt_dir / "adapter_model.safetensors"
checks.append(("adapter_model.safetensors exists", adapter_file.exists()))

print(f"\n--- Checklist Results ---")
for desc, ok in checks:
    icon = "✅" if ok else "❌"
    print(f"{icon}  {desc}")

if all(ok for _, ok in checks):
    print("\n🎉 All checks passed. You can proceed to statistical analysis.")
else:
    print("\n⚠️ Some checks failed. Please inspect them.")

## Step 6: Paired statistical tests (vs Baseline + vs LoRA)

Following the HANDOFF.md § 8 standard interface, load baseline and LoRA per-example scores for paired bootstrap and McNemar tests.

In [ ]:
from src.evaluation.statistical_tests import (
    bootstrap_ci, paired_bootstrap_ci_diff, mcnemar_test
)
import json
from pathlib import Path

# Scores for the current experiment
with open(ckpt_dir / "per_example_scores.json") as f:
    me = json.load(f)

# Baseline scores (Member 1 must already have been pushed to the repository)
baseline_path = Path("results/baseline/per_example_scores.json")
if not baseline_path.exists():
    print("⚠️ baseline scores do not exist; please confirm that Member 1 has pushed them.")
else:
    with open(baseline_path) as f:
        base = json.load(f)
    
    # Check paired consistency (HANDOFF.md § 8 requires identical ids)
    assert me['ids'] == base['ids'], "❌ Per-example IDs do not match! paired tests are invalid."
    print(f"✅ {len(me['ids'])} sample IDs match\n")
    
    print("=" * 70)
    print(f"{EXPERIMENT}  vs  Baseline (zero-shot)")
    print("=" * 70)
    
    # Overall EM
    diff = paired_bootstrap_ci_diff(
        base['correct'], me['correct'], n_resamples=10000, seed=42
    )
    mc = mcnemar_test(base['correct'], me['correct'])
    print(f"\nOverall EM (n=451):")
    print(f"  Δ = {diff['point']:+.4f}  95% CI [{diff['lower']:+.4f}, {diff['upper']:+.4f}]  p={diff['p_two_sided']:.4g}")
    print(f"  McNemar: b={mc['b']}, c={mc['c']}, p={mc['p_value']:.4g}")
    
    # Closed EM
    cl_idx = [i for i, q in enumerate(base['qtypes']) if q == 'closed']
    a = [base['correct'][i] for i in cl_idx]
    b = [me['correct'][i] for i in cl_idx]
    diff_c = paired_bootstrap_ci_diff(a, b, n_resamples=10000, seed=42)
    print(f"\nClosed EM (n={len(cl_idx)}):")
    print(f"  Δ = {diff_c['point']:+.4f}  95% CI [{diff_c['lower']:+.4f}, {diff_c['upper']:+.4f}]  p={diff_c['p_two_sided']:.4g}")
    
    # Open Token-F1
    op_idx = [i for i, q in enumerate(base['qtypes']) if q == 'open']
    a = [base['f1'][i] for i in op_idx]
    b = [me['f1'][i] for i in op_idx]
    diff_o = paired_bootstrap_ci_diff(a, b, n_resamples=10000, seed=42)
    print(f"\nOpen Token-F1 (n={len(op_idx)}):")
    print(f"  Δ = {diff_o['point']:+.4f}  95% CI [{diff_o['lower']:+.4f}, {diff_o['upper']:+.4f}]  p={diff_o['p_two_sided']:.4g}")

In [ ]:
# vs LoRA (matched rank; test whether QLoRA / DoRA preserves accuracy after quantization/decomposition)
import re

# Automatically find the LoRA checkpoint with the matched rank
rank_match = re.search(r'rank(\d+)', EXPERIMENT)
if rank_match:
    rank = rank_match.group(1)
    lora_path = Path(f"checkpoints/lora_rank{rank}/per_example_scores.json")
    
    if not lora_path.exists():
        print(f"⚠️ {lora_path} does not exist; please confirm that Member 1 has pushed this LoRA checkpoint.")
    else:
        with open(lora_path) as f:
            lora = json.load(f)
        assert me['ids'] == lora['ids']
        
        print("=" * 70)
        print(f"{EXPERIMENT}  vs  LoRA r={rank}")
        print("=" * 70)
        
        # Overall EM
        diff = paired_bootstrap_ci_diff(
            lora['correct'], me['correct'], n_resamples=10000, seed=42
        )
        mc = mcnemar_test(lora['correct'], me['correct'])
        print(f"\nOverall EM:")
        print(f"  Δ = {diff['point']:+.4f}  95% CI [{diff['lower']:+.4f}, {diff['upper']:+.4f}]  p={diff['p_two_sided']:.4g}")
        print(f"  McNemar: b={mc['b']}, c={mc['c']}, p={mc['p_value']:.4g}")
        
        # Interpretation
        if 'qlora' in EXPERIMENT.lower():
            if abs(diff['point']) < 0.03:
                print(f"\n  ✅ |Δ| < 3pp - consistent with Dettmers 2023 expectation (QLoRA should be nearly lossless)")
            else:
                print(f"\n  ⚠️ |Δ| >= 3pp - deviates from expectation; quantization settings may need debugging")
        elif 'dora' in EXPERIMENT.lower():
            if diff['point'] >= 0:
                print(f"\n  ✅ DoRA >= LoRA - consistent with the Liu 2024 paper expectation")
            else:
                print(f"\n  ℹ️ DoRA is slightly below LoRA; this may be true for this 1,797-sample dataset")

## Step 7: Sanity range check (HANDOFF.md § 9)

Compare against the expected ranges in HANDOFF.md § 9 to check for bugs.

In [ ]:
with open(ckpt_dir / "training_metrics.json") as f:
    m = json.load(f)

ev = m['evaluation']
closed_em = ev['closed']['exact_match']
open_f1 = ev['open']['f1']
overall_em = ev['overall']['exact_match']
peak_gpu = m['peak_gpu_memory_gb']

print("=" * 60)
print(f"Sanity Check: {EXPERIMENT}")
print("=" * 60)

is_qlora = 'qlora' in EXPERIMENT.lower()
is_dora = 'dora' in EXPERIMENT.lower()

if is_qlora:
    print("\n--- QLoRA Expected range (HANDOFF.md § 9) ---")
    print(f"Closed EM:      {closed_em:.4f}  (expected 0.65-0.78)  {'✅' if 0.65 <= closed_em <= 0.78 else '⚠️'}")
    print(f"Open Token-F1:  {open_f1:.4f}  (expected 0.25-0.38)  {'✅' if 0.25 <= open_f1 <= 0.38 else '⚠️'}")
    print(f"Overall EM:     {overall_em:.4f}  (expected 0.45-0.58)  {'✅' if 0.45 <= overall_em <= 0.58 else '⚠️'}")
    print(f"Peak GPU:       {peak_gpu:.2f} GB  (expected 3.5-4.5)    {'✅' if 3.5 <= peak_gpu <= 4.5 else '⚠️'}")
    if peak_gpu > 6.0:
        print(f"  ❌ Red flag: GPU > 6 GB indicates that 4-bit quantization is not actually active!")
elif is_dora:
    print("\n--- DoRA Expected range (HANDOFF.md § 9) ---")
    print(f"Closed EM:      {closed_em:.4f}  (expected 0.70-0.80)  {'✅' if 0.70 <= closed_em <= 0.80 else '⚠️'}")
    print(f"Open Token-F1:  {open_f1:.4f}  (expected 0.30-0.40)  {'✅' if 0.30 <= open_f1 <= 0.40 else '⚠️'}")
    print(f"Overall EM:     {overall_em:.4f}  (expected 0.50-0.60)  {'✅' if 0.50 <= overall_em <= 0.60 else '⚠️'}")
    print(f"Peak GPU:       {peak_gpu:.2f} GB  (expected 7.0-8.0)    {'✅' if 7.0 <= peak_gpu <= 8.0 else '⚠️'}")

# Red flag
print(f"\n--- Red-flag check ---")
red_flags = []
if closed_em < 0.55:
    red_flags.append(f"❌ Closed EM ({closed_em:.4f}) < 0.55 - worse than baseline!")
if open_f1 < 0.18:
    red_flags.append(f"❌ Open F1 ({open_f1:.4f}) < 0.18 - worse than baseline!")
if peak_gpu > 14:
    red_flags.append(f"❌ Peak GPU ({peak_gpu:.2f} GB) > 14 GB - should not be able to run on T4!")

if red_flags:
    for f in red_flags:
        print(f)
else:
    print("✅ No red flags")

## Step 8: Back up checkpoint to Google Drive (Important!)

Colab storage is cleared after the session ends, so back up first.

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy checkpoint to Drive
DRIVE_BACKUP = "/content/drive/MyDrive/peft_medical_vqa/checkpoints"
!mkdir -p {DRIVE_BACKUP}
!cp -r {ckpt_dir} {DRIVE_BACKUP}/
!ls -lh {DRIVE_BACKUP}/{EXPERIMENT}
print(f"\n✅ Backed up to {DRIVE_BACKUP}/{EXPERIMENT}")

## Step 9: Commit to git (following HANDOFF.md § 15 commit conventions)

In [ ]:
# Configure git (needed the first time)
!git config user.email "wd2423@columbia.edu"
!git config user.name "Wanrong Dang"

# Add checkpoint metrics, scores, and predictions to git (Note: adapter_model.safetensors excluded by .gitignore)
!git add configs/{EXPERIMENT}.yaml 2>/dev/null || true
!git add checkpoints/{EXPERIMENT}/training_metrics.json
!git add checkpoints/{EXPERIMENT}/per_example_scores.json
!git add checkpoints/{EXPERIMENT}/lora_predictions.jsonl

# Generate a commit message that follows the conventions
import re
method = 'qlora' if 'qlora' in EXPERIMENT.lower() else ('dora' if 'dora' in EXPERIMENT.lower() else 'sweep')
rank_match = re.search(r'rank(\d+)', EXPERIMENT)
rank = rank_match.group(1) if rank_match else '?'
msg = f"[member2] {method}: full-data ablation r={rank}\n\nClosed EM: {closed_em:.4f}, Open F1: {open_f1:.4f}, Overall EM: {overall_em:.4f}\nPeak GPU: {peak_gpu:.2f} GB, Trainable: {m['params']['trainable']:,}"
print("Commit message:")
print(msg)
print()

# Note: You must manually run the two lines below for the actual commit + push (to avoid accidental operations)
# !git commit -m "$(echo -e '{msg}')"
# !git push

---
## Done! Switch to the next experiment:

1. Change the `EXPERIMENT` variable in Step 3
2. Rerun all cells starting from Step 4
3. After completing all 6 experiments, switch to `cross_method_analysis.ipynb` for cross-method comparison